# new updates

In [ ]:
!pip install -q mlflow
!pip install -q --upgrade transformers datasets accelerate peft evaluate
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.0/688.0 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 19.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login
login()

In [ ]:
login(token = 'your_huggingface_token_here')

In [ ]:


# --- 1) Load & Split ---
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
print(f"Dataset size: {len(dataset)}")
print("Columns:", dataset.column_names)
print("Sample row:", dataset[0])

ds = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]
print(f"Train/Eval sizes: {len(train_ds)} / {len(eval_ds)}")

# --- 2) Format (your fmt exactly as written) ---
system_prompt = """You are a professional counselor and mental health advisor. Your role is to:

- Listen empathetically and provide supportive, evidence-based guidance
- Ask clarifying questions when needed to better understand the situation
- Offer practical coping strategies and actionable advice
- Recognize when professional help may be needed and provide appropriate referrals
- Maintain a warm, non-judgmental tone while being direct and helpful
- Focus on empowering the person to develop healthy coping mechanisms

Always prioritize the person's safety and well-being in your responses."""

BOS = "<|begin_of_text|>"
EOT = "<|eot_id|>"
EOS = "<|end_of_text|>"

def _role(r):
    return f"<|start_header_id|>{r}<|end_header_id|>\n\n"

def fmt(ex):
    context = str(ex["Context"]).strip()
    response = str(ex["Response"]).strip()
    enhanced_context = (
        "A person is seeking guidance with the following concern:\n\n"
        f"{context}\n\n"
        "Please provide thoughtful, professional advice."
    )
    text = (
        f"{BOS}"
        f"{_role('system')}{system_prompt}{EOT}"
        f"{_role('user')}{enhanced_context}{EOT}"
        f"{_role('assistant')}{response}{EOT}{EOS}"
    )
    return {"text": text}

formatted_train = train_ds.map(fmt, remove_columns=train_ds.column_names)
formatted_eval  = eval_ds.map(fmt,  remove_columns=eval_ds.column_names)
print("Formatted example:\n", formatted_train[0]["text"][:400])

# --- 3) Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Train tokenization (standard)
def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_train = formatted_train.map(tok_fn, batched=True, remove_columns=["text"])
print("Tokenized train features:", tokenized_train.features)

# --- 4) Assistant-only EVAL tokenization (mask non-assistant tokens) ---


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
import torch, math, mlflow, os

# Core config (tweak as needed)
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "./outputs-llama1b-counselor"
EXPERIMENT_NAME = "mental-health-chatbot"
RUN_NAME = "llama1b_counselor_sft"

MAX_LEN = 2048
EPOCHS = 3
LR = 1e-5
BSZ = 4
GRAD_ACCUM = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)



In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
print(f"Dataset size: {len(dataset)}")
print("Columns:", dataset.column_names)
print("Sample row:", dataset[0])

ds = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]
print(f"Train/Eval sizes: {len(train_ds)} / {len(eval_ds)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset size: 3512
Columns: ['Context', 'Response']
Sample row: {'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that i

In [ ]:
system_prompt = """You are a professional counselor and mental health advisor. Your role is to:

- Listen empathetically and provide supportive, evidence-based guidance
- Ask clarifying questions when needed to better understand the situation
- Offer practical coping strategies and actionable advice
- Recognize when professional help may be needed and provide appropriate referrals
- Maintain a warm, non-judgmental tone while being direct and helpful
- Focus on empowering the person to develop healthy coping mechanisms

Always prioritize the person's safety and well-being in your responses."""

BOS = "<|begin_of_text|>"
EOT = "<|eot_id|>"
EOS = "<|end_of_text|>"

def _role(r):
    return f"<|start_header_id|>{r}<|end_header_id|>\n\n"

def fmt(ex):
    context = str(ex["Context"]).strip()
    response = str(ex["Response"]).strip()

    enhanced_context = (
        "A person is seeking guidance with the following concern:\n\n"
        f"{context}\n\n"
        "Please provide thoughtful, professional advice."
    )

    text = (
        f"{BOS}"
        f"{_role('system')}{system_prompt}{EOT}"
        f"{_role('user')}{enhanced_context}{EOT}"
        f"{_role('assistant')}{response}{EOT}{EOS}"
    )
    return {"text": text}

formatted_train = train_ds.map(fmt, remove_columns=train_ds.column_names)
formatted_eval  = eval_ds.map(fmt,  remove_columns=eval_ds.column_names)

print("Formatted example:\n", formatted_train[0]["text"][:400])

Map:   0%|          | 0/3160 [00:00<?, ? examples/s]

Map:   0%|          | 0/352 [00:00<?, ? examples/s]

Formatted example:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a professional counselor and mental health advisor. Your role is to:

- Listen empathetically and provide supportive, evidence-based guidance
- Ask clarifying questions when needed to better understand the situation  
- Offer practical coping strategies and actionable advice
- Recognize when professional help may be needed and pr


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_train = formatted_train.map(tok_fn, batched=True, remove_columns=["text"])
#tokenized_eval  = formatted_eval.map(tok_fn,  batched=True, remove_columns=["text"])

print("Tokenized keys:", tokenized_train.features)




tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/3160 [00:00<?, ? examples/s]

Tokenized keys: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [ ]:
ASSISTANT_HDR = "<|start_header_id|>assistant<|end_header_id|>\n\n"
ah_ids  = tokenizer(ASSISTANT_HDR, add_special_tokens=False)["input_ids"]
eot_ids = tokenizer(EOT,           add_special_tokens=False)["input_ids"]

def _find_subseq(h, n):
    L, N = len(h), len(n)
    for i in range(L - N + 1):
        if h[i:i+N] == n:
            return i
    return None

def tok_fn_assistant_only(batch):
    out = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN)
    labels = []
    for ids in out["input_ids"]:
        lab = ids.copy()
        start = _find_subseq(ids, ah_ids)
        if start is None:
            lab[:] = [-100] * len(lab)
        else:
            s = start + len(ah_ids)
            rel_end = _find_subseq(ids[s:], eot_ids)
            e = s + rel_end if rel_end is not None else len(ids)
            for i in range(0, s):        lab[i] = -100
            for i in range(e, len(lab)): lab[i] = -100
        labels.append(lab)
    out["labels"] = labels
    return out

tokenized_eval = formatted_eval.map(tok_fn_assistant_only, batched=True, remove_columns=["text"])
print("Assistant-only eval prepared:", len(tokenized_eval), "examples")

Map:   0%|          | 0/352 [00:00<?, ? examples/s]

Assistant-only eval prepared: 352 examples


In [ ]:
# %% ------------------ Model ------------------
dtype = (
    torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else (torch.float16 if torch.cuda.is_available() else torch.float32)
)
print("Using dtype:", dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32, #dtype,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id

# %% ------------------ Collator ------------------
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Using dtype: torch.bfloat16


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
# %% ------------------ Training Args (MLflow integrated) ------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BSZ,
    per_device_eval_batch_size=BSZ,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.0,

    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,

    fp16=False, #(dtype==torch.float16),
    bf16=True, #(dtype==torch.bfloat16),
    gradient_checkpointing=True,
    report_to=["mlflow"],             # <-- key line for Transformers->MLflow logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collator,
)

# %% ------------------ Train + Evaluate with MLflow ------------------
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    # Helpful params in the run
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("max_len", MAX_LEN)
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("per_device_bsz", BSZ)
    mlflow.log_param("grad_accum", GRAD_ACCUM)
    mlflow.log_param("dtype", str(dtype))

    print("Starting training...")
    train_result = trainer.train()
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)

    print("Evaluating...")
    eval_metrics = trainer.evaluate()
    # Add perplexity for convenience
    if "eval_loss" in eval_metrics and eval_metrics["eval_loss"] is not None:
        eval_metrics["eval_perplexity"] = float(math.exp(eval_metrics["eval_loss"]))
    print("Eval metrics:", eval_metrics)

    # Log and save
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)
    trainer.save_state()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Done. MLflow runs saved under ./mlruns. Launch UI with: mlflow ui --port 5000")

2025/08/17 20:06:02 INFO mlflow.tracking.fluent: Experiment with name 'mental-health-chatbot' does not exist. Creating a new experiment.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Starting training...


Step,Training Loss,Validation Loss
200,1.337600,1.223645
400,0.821400,1.000475


***** train metrics *****
  epoch                    =        3.0
  total_flos               = 31074055GF
  train_loss               =       0.99
  train_runtime            = 0:13:15.03
  train_samples_per_second =     11.924
  train_steps_per_second   =      0.747
Evaluating...


Eval metrics: {'eval_loss': 0.9009296894073486, 'eval_runtime': 19.1413, 'eval_samples_per_second': 18.39, 'eval_steps_per_second': 4.597, 'epoch': 3.0, 'eval_perplexity': 2.4618908413901783}
***** eval metrics *****
  epoch                   =        3.0
  eval_loss               =     0.9009
  eval_perplexity         =     2.4619
  eval_runtime            = 0:00:19.14
  eval_samples_per_second =      18.39
  eval_steps_per_second   =      4.597
✅ Done. MLflow runs saved under ./mlruns. Launch UI with: mlflow ui --port 5000


In [ ]:
!zip -r /outputs-llama1b-counselor.zip /content/outputs-llama1b-counselor

  adding: content/outputs-llama1b-counselor/ (stored 0%)
  adding: content/outputs-llama1b-counselor/chat_template.jinja (deflated 71%)
  adding: content/outputs-llama1b-counselor/config.json (deflated 53%)
  adding: content/outputs-llama1b-counselor/trainer_state.json (deflated 72%)
  adding: content/outputs-llama1b-counselor/train_results.json (deflated 37%)
  adding: content/outputs-llama1b-counselor/tokenizer_config.json (deflated 96%)
  adding: content/outputs-llama1b-counselor/tokenizer.json (deflated 85%)
  adding: content/outputs-llama1b-counselor/model.safetensors (deflated 7%)
  adding: content/outputs-llama1b-counselor/special_tokens_map.json (deflated 63%)
  adding: content/outputs-llama1b-counselor/checkpoint-400/ (stored 0%)
  adding: content/outputs-llama1b-counselor/checkpoint-400/rng_state.pth (deflated 25%)
  adding: content/outputs-llama1b-counselor/checkpoint-400/optimizer.pt (deflated 9%)
  adding: content/outputs-llama1b-counselor/checkpoint-400/chat_template.jinj

In [ ]:
# Pack only inference essentials from your save dir, then download the zip.
import os, zipfile, pathlib
from google.colab import files

MODEL_DIR = "/content/outputs-llama1b-counselor"   # <- change if yours is different
ARCHIVE   = "/content/counselor_inference_only.zip"

# Always-try list (added only if present)
KEEP = {
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "generation_config.json",     # optional but handy
    "chat_template.jinja",        # optional if you want apply_chat_template
    "tokenizer.model",            # SentencePiece (some models)
    "spiece.model",               # SentencePiece (older name)
    "merges.txt", "vocab.json",   # BPE (some tokenizers)
    "added_tokens.json",          # if you added any special tokens
}

# Also include any safetensors shards (handles single or sharded weights)
def list_weight_files(root):
    out = []
    for f in os.listdir(root):
        if f.endswith(".safetensors") or f.endswith(".safetensors.index.json"):
            out.append(f)
    return sorted(out)

weight_files = list_weight_files(MODEL_DIR)
if not weight_files:
    raise FileNotFoundError("No *.safetensors weights found in MODEL_DIR")

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as z:
    # weights
    for f in weight_files:
        z.write(os.path.join(MODEL_DIR, f),
                arcname=os.path.join("outputs-llama1b-counselor", f))
    # configs/tokenizer/etc.
    for f in KEEP:
        p = os.path.join(MODEL_DIR, f)
        if os.path.exists(p):
            z.write(p, arcname=os.path.join("outputs-llama1b-counselor", f))

print("Wrote:", ARCHIVE)
files.download(ARCHIVE)

Wrote: /content/counselor_inference_only.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>